<img src="https://cdn.jsdelivr.net/gh/maxischa/datacamp_test@bf9431e/ressources/img/logo_macmia.png" alt="Banque des Territoires · France 2030 · MACMIA" width="520">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc4_ml_v2/exercices/seance2_exercices.ipynb)

# Séance 4.2 — Prédire une décision — qui va résilier ?

**Exercices** · durée : 4h (2h de cours, 2h d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- préparer des données pour un problème de **classification**
- construire un modèle qui estime une **probabilité de résiliation**
- transformer cette probabilité en décision à l'aide d'un **seuil**
- comprendre les différentes erreurs possibles grâce à la **matrice de confusion**
- choisir une mesure d'évaluation adaptée parmi la justesse, la précision, le rappel et le F1
- comprendre pourquoi le choix du seuil dépend du **contexte métier** et du coût des erreurs

## Exercice — Le radar du pôle prestige

## La mission

L'agence a un **pôle prestige** : trois experts qui traitent les biens à plus
d'un million d'euros. Ils les estiment autrement, les photographient
autrement, les vendent autrement.

Le problème est en amont. Un mandat arrive avec une adresse, une surface et
un nombre de pièces — **pas un prix**, c'est justement ce qu'il faut
déterminer. Et il faut décider tout de suite qui s'en occupe.

Se tromper coûte dans les deux sens :

| L'erreur | Ce qu'elle coûte |
|---|---|
| envoyer un studio à l'expert prestige | une visite pour rien : **200 €** |
| laisser un bien d'exception au pôle standard | sous-estimé, mal vendu, souvent perdu : **4 000 €** |

La direction veut un radar. Vous allez le construire, puis décider **où
placer le curseur** — et c'est cette seconde décision qui vaudra le plus
cher.

Ensuite, la direction vous demandera un second modèle. Celui-là ne marchera
pas, et comprendre pourquoi est la partie la plus utile de l'exercice.

## Comment ça marche

**Chaque exercice est une cellule vide que vous écrivez entièrement.** Juste
avant, un encadré *Rappel* nomme les outils dont vous avez besoin.

Cet exercice est plus difficile que les précédents, et il est construit pour
que vous ne restiez jamais bloqué :

- la **mécanique** — les appels à scikit-learn qui ne s'inventent pas — vous est donnée ;
- ce que vous écrivez, ce sont les **choix** : quelles colonnes, quelle mesure, quel verdict ;
- les **cellules de pari** vous demandent d'annoncer un résultat *avant* de l'exécuter. Elles ne sont pas décoratives : c'est en se trompant de pari qu'on apprend à lire un chiffre ;
- les **cellules de vérification** affichent `OK` ou `A REVOIR` avec un indice, aux endroits où une erreur fausserait la suite.

Si une vérification affiche `NameError`, c'est que la cellule d'exercice
au-dessus n'a pas été exécutée, ou qu'elle contient une faute. Corrigez-la,
relancez-la, puis relancez la vérification.

## Partie 0 — Mise en route

La cellule de setup a une ligne de plus que celle de l'exercice précédent :
`LogisticRegression`, et les mesures de la classification.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://cdn.jsdelivr.net/gh/maxischa/datacamp_test@bf9431e/bloc2_donnees_v2/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

In [ ]:
ventes = pd.read_csv(BASE + "immo_paris_2024.csv")

print(ventes.shape)
ventes.head(3)

---

## Partie 1 — La cible, et ce qu'on a le droit de regarder

### Exercice 1 — Fabriquer la cible

Un bien d'exception, pour l'agence, c'est une vente à **plus d'un million
d'euros**. Construisez `y` : 1 pour un bien d'exception, 0 sinon. Puis
affichez le taux de biens d'exception du fichier, en pourcentage.

> **Rappel.** Une comparaison rend des Vrai/Faux ; `.astype(int)` les
> transforme en 1 et 0. La **moyenne** d'une colonne de 0 et de 1 est la
> proportion de 1 — c'est le calcul du taux de résiliation du cours 4.2.

In [ ]:
verifier("la cible", y.sum() == 3077, "prix strictement superieur a 1 000 000, puis astype(int)")
verifier("le taux", round(100 * y.mean(), 1) == 12.2, "la moyenne d'une colonne de 0 et de 1")

**12,2 % de biens d'exception.** Retenez ce nombre : il va piéger la
première mesure que vous calculerez.

### Exercice 2 — Ce que le radar a le droit de savoir

Trois questions, à répondre en commentaire **avant** de choisir les
variables. Elles décident de tout le reste.

1. *Le jour où le mandat arrive, connaît-on le prix ?*
2. *`prix` a-t-il le droit d'entrer dans `X` ?*
3. *Et `prix_m2` ?*

Il reste donc `surface`, `pieces` et `arrondissement`. Trois colonnes, c'est
peu — et c'est exactement ce dont dispose la personne qui ouvre le courrier.

### Le duel

Avant de laisser un modèle décider, essayez. Voici six mandats réels, avec
les seules informations connues à l'arrivée.

In [ ]:
mandats = ventes.loc[[48, 1, 755, 6, 29, 85], ["surface", "pieces", "arrondissement"]]

mandats

Pour chacun, dans l'ordre du tableau : **1** si vous l'envoyez au pôle
prestige, **0** sinon. Écrivez vos six réponses dans la liste.

Vous les comparerez à celles du modèle à la fin de la partie 3. Jouez le jeu :
ne regardez pas les prix.

In [ ]:
verifier("six reponses", len(mes_reponses) == 6, "une par mandat, dans l'ordre du tableau")

### Votre fiche de décision

Complétez cette cellule de texte (double-clic pour l'éditer). Vous la
relirez à la partie 4, au moment de régler le curseur.

- Ce que je prédis : …
- Ce que je connais au moment de décider : …
- Un bien d'exception raté coûte … ; une visite pour rien coûte … .
- Donc l'erreur que je veux surtout éviter est : …

---

## Partie 2 — Le radar, et le piège de la justesse

### Exercice 3 — Préparer les variables

Construisez `X` avec `surface`, `pieces`, et l'arrondissement transformé en
colonnes de 0 et de 1.

> **Rappel.** `pd.get_dummies(colonne, prefix="arr", drop_first=True)` fait la
> transformation ; `pd.concat([..., ...], axis=1)` recolle les morceaux côte à
> côte. C'est la préparation du cours 4.2 — et elle est nécessaire parce que
> l'arrondissement est une **catégorie**, pas une quantité : le 16e n'est pas
> « huit fois » le 2e.

In [ ]:
verifier("les colonnes de X", X.shape[1] == 21, "surface, pieces, et 19 arrondissements apres drop_first")
verifier("pas de fuite", "prix" not in X.columns and "prix_m2" not in X.columns,
         "ni le prix ni le prix au m2 n'ont le droit d'etre la")

### Exercice 4 — Découper, puis entraîner

Découpez avec `test_size=0.25`, `random_state=67` et **`stratify=y`**.

`stratify=y` garde le même taux de biens d'exception des deux côtés. Avec
seulement 12,2 % de positifs, un tirage malheureux pourrait en mettre
beaucoup plus d'un côté que de l'autre, et la note ne voudrait plus rien
dire.

> **Rappel.** `train_test_split(X, y, test_size=..., random_state=..., stratify=y)`.

In [ ]:
verifier("le jeu de test", len(y_test) == 6303, "un quart des ventes")
verifier("stratify a fait son travail", round(100 * y_train.mean(), 1) == round(100 * y_test.mean(), 1),
         "les deux taux doivent etre identiques a une decimale pres")

La cellule suivante entraîne le radar. C'est le pipeline du cours 4.2 :
`StandardScaler` met les variables à la même échelle — la surface va de 9 à
1 500, les colonnes d'arrondissement valent 0 ou 1 — puis la régression
logistique estime ses coefficients.

In [ ]:
radar = make_pipeline(StandardScaler(), LogisticRegression())
radar.fit(X_train, y_train)

# predict_proba rend deux colonnes : [proba ordinaire, proba exception]
proba = radar.predict_proba(X_test)[:, 1]

print("probabilite d'etre un bien d'exception, cinq premiers mandats :", proba[:5].round(3))

### Exercice 5 — La première note

Calculez la justesse du radar sur le jeu de test, dans `justesse`.

> **Rappel.** `radar.predict(X_test)` tranche à 0,50 et rend des 0 et des 1.
> `accuracy_score(reel, predit)` compare.

In [ ]:
verifier("la justesse", round(100 * justesse, 1) == 95.2, "accuracy_score(y_test, radar.predict(X_test))")

**95,2 % de bonnes réponses.** De quoi présenter le projet en comité.

Avant ça, une question. Écrivez votre pari :

In [ ]:
# Mon pari : un modele qui repondrait TOUJOURS "bien ordinaire",
# sans jamais rien regarder, aurait une justesse de ....... %

### Exercice 6 — Le modèle nul

Calculez-la. Un modèle qui répond toujours 0 a raison chaque fois que la
vraie réponse est 0.

> **Rappel.** La proportion de 0 dans `y_test`, c'est 1 moins la proportion
> de 1 — et la proportion de 1 est la moyenne de la colonne.

In [ ]:
verifier("le modele nul", round(100 * justesse_nulle, 1) == 87.8, "1 - y_test.mean()")

**87,8 % sans rien faire.** Notre radar ne gagne que
**7,4 points** sur un modèle qui n'enverrait jamais personne au
pôle prestige — et qui ne sauverait donc aucun mandat.

C'est le piège des données **déséquilibrées** du cours 4.2, et il est plus
sévère ici que sur le churn : quand une classe pèse 87,8 % du fichier,
la justesse récompense le fait de toujours parier dessus.

> ⚠️ **Une justesse ne se lit jamais seule.** Toujours à côté du score du
> modèle nul.

---

## Partie 3 — Les deux façons de se tromper

### Exercice 7 — La matrice de confusion

Affichez la matrice de confusion du radar, avec des étiquettes lisibles.

> **Rappel.** `confusion_matrix(y_test, prediction)` rend un tableau de
> quatre nombres. Pour le lire, `pd.DataFrame(..., index=[...], columns=[...])`
> avec les quatre libellés : en lignes la vérité, en colonnes la prédiction.

In [ ]:
verifier("les biens d'exception rates", matrice.loc["exception en vrai", "predit ordinaire"] == 208,
         "la ligne 'exception en vrai', colonne 'predit ordinaire' : les faux negatifs")
verifier("les visites pour rien", matrice.loc["ordinaire en vrai", "predit exception"] == 96,
         "la ligne 'ordinaire en vrai', colonne 'predit exception' : les faux positifs")

|  | prédit ordinaire | prédit exception |
|---|---:|---:|
| **ordinaire en vrai** | 5 438 | 96 — visites pour rien |
| **exception en vrai** | **208 — biens perdus** | 561 |

### Exercice 8 — Les quatre mesures

Calculez la précision, le rappel et le F1.

> **Rappel.** `precision_score`, `recall_score`, `f1_score`, tous avec
> `(y_test, prediction)`. La précision se divise par ceux qu'on **envoie** à
> l'expert, le rappel par ceux qui **sont** des biens d'exception.

In [ ]:
verifier("la precision", round(precision, 3) == 0.854, "precision_score(y_test, prediction)")
verifier("le rappel", round(rappel, 3) == 0.73, "recall_score(y_test, prediction)")

**Rappel 0,73.** Traduit en français : sur 769 biens d'exception
du jeu de test, **le radar en laisse 208 partir au pôle standard**.

Chiffrons-les tout de suite, puisque l'agence connaît le prix de cette
erreur : 208 × 4 000 = **832 000 €**.

Un modèle à 95,2 % de justesse vient de coûter 832 000 € à l'agence.
La justesse ne le disait pas. Le rappel, si.

La cellule suivante montre pourquoi. Elle place les 6 303 mandats du jeu
de test selon la probabilité que le radar leur donne, en séparant les deux
populations.

In [ ]:
pr = pd.Series(proba, index=y_test.index)

plt.figure(figsize=(7, 4))
pr[y_test == 0].plot(kind="hist", bins=50, alpha=0.7, color="grey", label="biens ordinaires")
pr[y_test == 1].plot(kind="hist", bins=50, alpha=0.7, color="#D64541", label="biens d'exception")
plt.axvline(0.5, color="black", linewidth=2, label="seuil par defaut : 0,50")
plt.yscale("log")   ## sans l'echelle log, les 769 biens d'exception disparaissent sous les autres
plt.xlabel("probabilite estimee d'etre un bien d'exception")
plt.ylabel("nombre de mandats (echelle log)")
plt.title("Le radar separe bien - mais le trait noir coupe au mauvais endroit")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

Les deux tas sont **bien séparés** : le radar a appris quelque chose. Mais
regardez ce que le trait noir laisse à sa gauche : tous les biens d'exception
rouges qui vivent sous 0,50, et que personne n'ira voir.

Rien n'oblige à couper là. C'est l'objet de la partie 4.

### Le duel, le verdict

Vos six réponses contre celles du radar.

In [ ]:
verite = (ventes.loc[mandats.index, "prix"] > 1_000_000).astype(int)
proba_mandats = radar.predict_proba(X.loc[mandats.index])[:, 1]

duel = mandats.copy()
duel["votre reponse"] = mes_reponses
duel["le radar"] = (proba_mandats > 0.5).astype(int)
duel["proba du radar"] = proba_mandats.round(2)
duel["la verite"] = verite.values
duel["prix reel"] = ventes.loc[mandats.index, "prix"].values

print("votre score :", (np.array(mes_reponses) == verite.values).sum(), "/ 6")
print("le radar    :", ((proba_mandats > 0.5).astype(int) == verite.values).sum(), "/ 6")
duel

Regardez les mandats où la probabilité tourne autour de 0,5 : ce sont ceux
sur lesquels vous avez hésité, vous aussi. Ce n'est pas un hasard. **Le radar
n'a rien de plus que vous** — une surface, un nombre de pièces, un
arrondissement. Là où ces trois informations ne suffisent pas à trancher, ni
lui ni vous ne trancherez.

---

## Partie 4 — Le curseur, et ce qu'il vaut

`predict` a tranché à 0,50. Ce nombre ne vient d'aucune réflexion sur votre
métier : c'est un réglage par défaut. Reprenons-le.

### Exercice 9 — Descendre le seuil

Pour chacun des seuils 0,50, 0,40, 0,30, 0,20 et 0,10, affichez le nombre de
mandats signalés, le rappel et la précision.

> **Rappel.** `(proba > seuil).astype(int)` fabrique la décision, `.sum()`
> compte les signalés — c'est le `(proba > 0.20).astype(int)` du cours 4.2.
> Une boucle `for seuil in [...]` évite de recopier quatre fois.

In [ ]:
verifier("le seuil 0,10", (proba > 0.10).sum() == 1238, "(proba > 0.10).sum() compte les True")

Descendre le seuil rattrape des biens d'exception (le rappel monte) au prix
de visites inutiles (la précision baisse). Les deux ne montent jamais
ensemble.

**Aucun de ces réglages n'est « le bon » en soi.** Pour trancher, il faut
sortir des mesures et regarder les euros.

La cellule suivante écrit le calcul une fois pour toutes, comme la fonction
`gain` du cours 4.2.

In [ ]:
def cout(seuil, prix_visite=200, prix_mandat_perdu=4000):
    """Ce que coutent les erreurs du radar, a ce seuil, en euros."""
    decision = (proba > seuil).astype(int)
    rates = ((decision == 0) & (y_test == 1)).sum()     ## biens d'exception manques
    vaines = ((decision == 1) & (y_test == 0)).sum()    ## visites pour rien
    return rates * prix_mandat_perdu + vaines * prix_visite

print("au seuil par defaut de 0,50 :", cout(0.50), "euros")

### Exercice 10 — Les deux stratégies sans modèle

Avant de régler quoi que ce soit : **combien coûteraient les deux décisions
qui ne demandent aucun modèle ?**

- `cout_rien` : ne rien faire, tout envoyer au pôle standard. Aucune visite
  inutile, mais tous les biens d'exception sont ratés.
- `cout_tout` : envoyer tous les mandats à l'expert. Aucun bien raté, mais
  une visite pour chaque bien ordinaire.

> **Rappel.** Le nombre de biens d'exception du jeu de test est
> `y_test.sum()` ; le nombre de biens ordinaires, `(y_test == 0).sum()`.

In [ ]:
verifier("ne rien faire", cout_rien == 3076000, "tous les biens d'exception rates, a 4 000 euros piece")
verifier("tout faire visiter", cout_tout == 1106800, "une visite a 200 euros par bien ordinaire")

**3 076 000 €** si on ne fait rien, **1 106 800 €** si on envoie tout
à l'expert, **851 200 €** avec le radar réglé par défaut.

Le radar bat les deux. C'est le minimum qu'on lui demande — et remarquez
qu'on vient de le juger **en euros**, pas en F1.

### Exercice 11 — Trouver le bon curseur

Parcourez la grille de seuils ci-dessous, affichez le coût de chacun, et
mettez le meilleur dans `meilleur_seuil`.

> **Rappel.** Une liste vide, une boucle, `append` : le réflexe du bloc 3. Une
> fois les coûts dans une Series indexée par les seuils, `idxmin()` rend
> l'étiquette du plus petit.

In [ ]:
verifier("le meilleur seuil", meilleur_seuil == 0.05, "idxmin() rend l'etiquette, pas la valeur")
verifier("le cout a l'optimum", couts.min() == 237600, "cout(seuil) pour chaque seuil de la grille")

**237 600 € au seuil de 0,05, contre 851 200 € au seuil par
défaut.**

**613 600 € d'écart, et pas une ligne du modèle n'a changé.** Les
probabilités sont les mêmes, les coefficients sont les mêmes. Seul le nombre
à partir duquel on décide d'agir a bougé.

> ⚠️ **Ne laissez jamais `predict` choisir à votre place.** Son 0,50 est une
> convention informatique. Dès qu'une erreur coûte plus cher que l'autre, le
> bon seuil se calcule.

In [ ]:
plt.figure(figsize=(7, 3.6))
plt.plot(grille, couts / 1000, color="#2878B5", marker="o")
plt.axvline(meilleur_seuil, color="#D64541",
            label=f"optimum {meilleur_seuil} : {couts.min() / 1000:.0f} k EUR")
plt.axvline(0.50, color="grey", linestyle="--",
            label=f"defaut 0,50 : {cout(0.50) / 1000:.0f} k EUR")
plt.xlabel("seuil de decision"); plt.ylabel("cout des erreurs (milliers d'euros)")
plt.title("Le meme radar, du simple au quadruple selon le curseur")
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()

### Le tableau de bord

Les deux prix étaient des hypothèses. Changez-les, et regardez le curseur se
déplacer tout seul.

In [ ]:
prix_visite = 200          ## changez-moi
prix_mandat_perdu = 4000   ## changez-moi

essai = []
for seuil in grille:
    essai.append(cout(seuil, prix_visite, prix_mandat_perdu))
essai = pd.Series(essai, index=grille)

print(f"visite {prix_visite} EUR, mandat perdu {prix_mandat_perdu} EUR")
print(f"  -> seuil optimal {essai.idxmin()} pour {essai.min():,} euros".replace(",", " "))
print(f"  -> {(proba > essai.idxmin()).sum()} mandats envoyes a l'expert sur {len(y_test)}")

Essayez une visite à 1 000 € : le curseur remonte, l'expert se déplace moins.
Essayez un mandat perdu à 20 000 € : il descend encore. **Le bon seuil n'est
pas une propriété du modèle, c'est une propriété de votre métier.**

### Exercice 12 — Décider

Deux réponses à écrire en commentaire.

1. *Quel seuil recommandez-vous, et combien de visites d'expert cela
   représente-t-il sur les 6 303 mandats du jeu de test ?*
2. *La direction impose « pas plus de 800 visites ». Quel seuil prenez-vous
   alors, et combien de biens d'exception acceptez-vous de perdre ?* La
   réponse est dans le tableau de l'exercice 9 et dans la cellule ci-dessus.

---

## Partie 5 — Le second modèle

La direction est contente. Elle en veut un autre :

> *« Puisque ça marche, faites-nous la même chose pour repérer les bonnes
> affaires — les biens qui partent sous le prix du quartier. On se
> positionnerait dessus. »*

Une bonne affaire, disons : une vente dont le prix au m² est sous le
**premier quartile de son propre arrondissement**. Un studio à 7 000 € le m²
est une affaire dans le 6e, pas dans le 19e.

La cellule suivante fabrique cette cible.

In [ ]:
# transform : comme un groupby, mais il RENVOIE une valeur par ligne au lieu
# d'une par groupe. Chaque vente recoit ainsi le q25 de SON arrondissement.
q25_arr = ventes.groupby("arrondissement")["prix_m2"].transform(lambda s: s.quantile(0.25))
affaire = (ventes["prix_m2"] < q25_arr).astype(int)

print(round(100 * affaire.mean(), 1), "% de bonnes affaires, par construction")

### Exercice 13 — Refaire exactement le même travail

Vous avez tout fait il y a une heure. Refaites-le en cinq minutes, avec
`affaire` à la place de `y` : découpage stratifié (`random_state=67`),
pipeline, entraînement, puis la justesse, la précision, le rappel et le F1.

Rangez les probabilités dans `proba_affaire`.

> **Rappel.** Les mêmes lignes que les exercices 4, 5, 7 et 8. Le même `X` :
> on ne connaît toujours que la surface, les pièces et l'arrondissement.

In [ ]:
verifier("la justesse du chasseur", round(accuracy_score(ya_test, pred_affaire), 3) == 0.75,
         "le meme travail qu'aux exercices 4 et 5, avec affaire a la place de y")
verifier("son F1", round(f1_score(ya_test, pred_affaire, zero_division=0), 3) == 0.0,
         "f1_score : il vaut vraiment zero, ce n'est pas une erreur de votre part")

```
justesse  : 0,75        modele nul : 0,75
precision : 0,0
rappel    : 0,0
F1        : 0,0
```

**La justesse est exactement celle du modèle nul.** Précision, rappel et F1
valent zéro. L'argument `zero_division=0` sert justement à ça : il dit à
scikit-learn de renvoyer 0 plutôt qu'une erreur quand une division n'a pas de
sens — et ici elle n'en a pas, parce qu'il n'y a **rien** à diviser.

### Exercice 14 — Comprendre ce qui vient de se passer

Deux questions. Écrivez le code, puis la réponse en commentaire.

1. *Combien de bonnes affaires le chasseur signale-t-il ?*
2. *Quelle est la plus forte probabilité qu'il attribue à un bien ?*

> **Rappel.** `pred_affaire.sum()` compte les 1. `.max()` sur
> `proba_affaire`.

In [ ]:
verifier("aucune affaire signalee", pred_affaire.sum() == 0, "predict tranche a 0,50")
verifier("la probabilite maximale", round(proba_affaire.max(), 2) == 0.33,
         "le maximum de proba_affaire : il ne franchit jamais 0,50")

### Exercice 15 — Et si on baissait le seuil ?

C'est le réflexe que vous venez d'acquérir en partie 4. Essayez-le : pour les
seuils 0,30, 0,25 et 0,20, affichez le nombre de signalés, la précision et le
rappel.

Et affichez à côté **la précision du hasard** : si on tirait des mandats au
sort, la proportion de bonnes affaires parmi eux serait le taux de base,
`ya_test.mean()`.

> **Rappel.** La même boucle qu'à l'exercice 9.

In [ ]:
verifier("la precision au seuil 0,25", round(precision_score(ya_test, (proba_affaire > 0.25).astype(int), zero_division=0), 3) == 0.271,
         "comparez-la au taux de base : ya_test.mean()")

Au seuil de 0,25, le chasseur signale 3 430 affaires avec une précision
de **0,271**. Le F1 remonte à 0,372 et pourrait presque faire
illusion.

Mais la dernière colonne tue l'illusion : en tirant des mandats **au hasard**,
on aurait une précision de **0,25**. Le modèle apporte
**2,1 points de précision sur un tirage au sort** — autrement dit, rien.

> **Baisser un seuil déplace un curseur. Ça ne crée pas d'information.**
> Quand il n'y a rien à trouver, aucun réglage ne le fera apparaître.

### Pourquoi celui-là ne pouvait pas marcher

Trois raisons, dans cet ordre.

1. **Le fichier ne contient ni l'étage, ni l'état, ni la vue, ni la raison de
   la vente.** C'est la phrase de la fin de l'exercice précédent : ce que le
   fichier ne contient pas, aucun modèle ne l'inventera.
2. **La cible est définie par rapport à son propre arrondissement.** Du coup
   l'arrondissement, qui était la variable la plus utile du radar, ne sert
   plus à rien ici : on a soigneusement retiré l'information que le modèle
   savait exploiter.
3. Et la raison de fond, qui vaut d'être retenue : **une bonne affaire est,
   par définition, ce que les caractéristiques visibles n'expliquent pas.** Si
   la surface et l'adresse suffisaient à la repérer, quelqu'un l'aurait déjà
   achetée. C'est exactement le **résidu** du modèle de l'exercice
   précédent — ce qui reste quand on a tout expliqué.

### Exercice 16 — Répondre à la direction

En trois phrases, en commentaire : ce que vous ne livrerez pas, pourquoi, et
**ce qu'il faudrait pour que ça marche**.

Savoir dire non à un projet, avec un chiffre à l'appui, est une compétence.
C'est peut-être la plus utile de tout le bloc.

---

## Partie 6 — Le modèle parfait

### Exercice 17 — La fuite

Revenons au radar, et donnons-lui le prix.

Construisez `X_fuite` en ajoutant la colonne `prix` à `X`, refaites le
découpage stratifié et l'entraînement, et mesurez la justesse.

> **Rappel.** `pd.concat([X, ventes[["prix"]]], axis=1)`, puis les mêmes
> lignes qu'à l'exercice 4.

In [ ]:
verifier("la justesse du tricheur", round(accuracy_score(yf_test, pred_fuite), 3) == 0.996,
         "ajoutez la colonne prix a X, rien d'autre")

**0,996 de justesse**, un rappel de 0,973. Le modèle parfait.

En commentaire : *pourquoi est-il parfait, et pourquoi est-il inutilisable ?*

---

## Pour conclure

### La note au directeur

Complétez cette cellule de texte :

- Pour le radar prestige, nous recommandons le seuil … , soit … visites d'expert et … € d'erreurs contre … € au réglage par défaut.
- Nous ne livrons pas le détecteur de bonnes affaires, parce que …
- Pour le relancer un jour, il faudrait …

### Ce que vous avez fait

- vous avez construit un radar qui trouve 73 % des biens d'exception, et vous avez surtout vu que sa justesse de 95,2 % ne valait que 7,4 points de mieux que ne rien faire ;
- vous avez transformé une probabilité en décision, et **trouvé 613 600 € dans un seul nombre** ;
- vous avez chiffré ce que coûte une contrainte de capacité ;
- vous avez construit un modèle qui ne sert à rien, compris pourquoi, et su le dire ;
- vous avez construit un modèle parfait, et compris pourquoi il est pire que le précédent.

| Vous avez utilisé | Pour |
|---|---|
| `(colonne > valeur).astype(int)` | fabriquer une cible binaire |
| `pd.get_dummies(..., drop_first=True)` | l'arrondissement, qui est une catégorie |
| `train_test_split(..., stratify=y)` | garder le même taux des deux côtés |
| `make_pipeline(StandardScaler(), LogisticRegression())` | le modèle du cours |
| `predict_proba(...)[:, 1]` | une probabilité plutôt qu'une décision |
| `confusion_matrix` et les quatre mesures | les deux façons de se tromper |
| `1 - y_test.mean()` | le modèle nul, à côté duquel toute justesse se lit |
| `(proba > seuil).astype(int)` | reprendre à `predict` une décision qui n'est pas la sienne |
| `def cout(seuil)` + `idxmin()` | régler le curseur en euros |
| `groupby().transform()` | une valeur par ligne, calculée dans son groupe |

### Les quatre phrases à retenir

1. **Une justesse ne se lit jamais sans le score du modèle nul.** 95,2 %
   contre 87,8 %, c'est 7,4 points, pas 95,2.
2. **Le seuil n'est pas dans le modèle, il est dans la décision.** Ici il
   valait 613 600 €.
3. **Un modèle qui n'a pas l'information ne la trouvera pas**, et baisser le
   seuil n'y change rien.
4. **Un score trop beau est un symptôme.** Cherchez la fuite avant de sabrer
   le champagne.

> ⚠️ **Avant de fermer l'onglet :** vérifiez que votre notebook est bien
> enregistré dans votre Drive.